# 12 techniques utiles de manipulation de données avec la librairie Pandas

Notebook de l'article [12 techniques utiles de manipulation de données avec la librairie Pandas](https://moncoachdata.com/blog/manipulation-de-donnees-avec-pandas/) (MonCoachData). Exécute les cellules dans l'ordre.

Python est le langage de prédilection des spécialistes des données, et ce pour de bonnes raisons. Il offre à la fois l'écosystème le plus vaste pour un langage de programmation et d'excellentes bibliothèques de calcul scientifique.

Parmi ces bibliothèques, Pandas est la plus utile pour les opérations de data science. Pandas associé à Scikit-learn, c'est le Graal.

Cet article présente 12 techniques de manipulation de données avec Pandas, avec quelques conseils et astuces pour travailler plus rapidement.

Pour t'aider à mieux comprendre, j'ai choisi un dataset sur lequel effectuer ces opérations : le problème de prédiction de prêt (*Loan Prediction*), dont le fichier `train.csv` (614 lignes) est chargé ici directement depuis le dépôt GitHub de MonCoachData.

**En bref :** ces 12 techniques Pandas couvrent l'essentiel de la manipulation de données : indexation booléenne, `apply`, imputation avec `fillna()`, `pivot_table`, multi-index, `crosstab`, `merge`, tri, graphiques directs, `cut`, codage des variables nominales et typage des colonnes. Elles sont appliquées au jeu de données Loan Prediction : par exemple, 80 % des personnes avec des antécédents de crédit obtiennent un prêt, contre 8 % des autres.

## Comment charger le dataset avec Pandas ?

Je commence par importer les modules et charger le dataset dans l'environnement Python :

In [ ]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/manipulation-de-donnees-avec-pandas/train.csv"

data = pd.read_csv(URL, index_col="Loan_ID")

## # 1 - Comment filtrer des lignes avec l'indexation booléenne ?

Pour filtrer des lignes selon des conditions sur plusieurs colonnes, utilise l'indexation booléenne : chaque condition entre parenthèses, combinées avec l'opérateur `&`. Que fais-tu si tu souhaites filtrer les valeurs d'une ou plusieurs colonnes en fonction de conditions portant sur d'autres colonnes ?

Par exemple, imaginons que l'on veuille la liste de toutes les femmes non diplômées qui ont obtenu un prêt. L'indexation booléenne est faite pour ça : chaque condition est placée entre parenthèses, et les conditions sont combinées avec l'opérateur `&`.

Tu peux utiliser le code suivant :

In [ ]:
data.loc[(data["Gender"]=="Female") & (data["Education"]=="Not Graduate") & (data["Loan_Status"]=="Y"), ["Gender","Education","Loan_Status"]]

(Pour aller plus loin : [documentation Indexing and Selecting Data](https://pandas.pydata.org/docs/user_guide/indexing.html))

## # 2 - À quoi sert la fonction apply ?

C'est l'une des principales fonctions pour manipuler les données et créer de nouvelles variables. `apply` applique une fonction à chaque ligne ou colonne d'un DataFrame et renvoie le résultat. Cette fonction peut être une fonction existante ou une fonction définie par l'utilisateur.

Par exemple, ici, `apply` permet de compter les valeurs manquantes de chaque colonne, puis de chaque ligne :

In [ ]:
# Créer une nouvelle fonction qui détermine si la valeur en paramètre est manquante:
def num_missing(x):
    return sum(x.isnull())

# On applique cette fonction pour chaque colonne:
print("Valeurs manquantes par colonne:")
print(data.apply(num_missing, axis=0)) #axis=0 définit que la fonction sera bien appliquée sur chaque colonne

# Puis application pour chaque ligne:
print("\nValeurs manquantes par ligne:")
print(data.apply(num_missing, axis=1).head()) #axis=1 définit que la fonction sera bien appliquée sur chaque ligne

On obtient bien le résultat souhaité.

Remarque : on utilise la méthode `head()` pour la deuxième application, car le résultat contenait beaucoup trop de lignes.

(Pour aller plus loin : [documentation apply de Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html))

## # 3 - Comment imputer les valeurs manquantes avec fillna() ?

La méthode `fillna()` le fait en une ligne. On l'utilise pour remplacer les valeurs manquantes par la moyenne (`mean`), le mode (`mode`) ou la médiane (`median`) de la colonne.

Imputons les colonnes `Gender` (sexe), `Married` (marié) et `Self_Employed` (à son compte) avec leur mode respectif. Commençons par calculer le mode de la colonne `Gender` :

In [ ]:
# Calculer le mode de la colonne
data['Gender'].mode()

La méthode `mode()` renvoie une Series, car plusieurs valeurs peuvent avoir la même fréquence maximale. On prend donc toujours le premier élément :

In [ ]:
data['Gender'].mode()[0]

Maintenant, on peut remplir les valeurs manquantes et vérifier le résultat avec la technique #2 :

In [ ]:
# Imputer les valeurs (on réassigne la colonne):
data['Gender'] = data['Gender'].fillna(data['Gender'].mode().iloc[0])
data['Married'] = data['Married'].fillna(data['Married'].mode().iloc[0])
data['Self_Employed'] = data['Self_Employed'].fillna(data['Self_Employed'].mode().iloc[0])

# Puis on re-teste les valeurs manquantes de nouveau pour valider:
print(data.apply(num_missing, axis=0))

Les valeurs manquantes ont bien été imputées. Note qu'il s'agit de la forme d'imputation la plus simple. Des techniques plus sophistiquées existent, comme la modélisation des valeurs manquantes ou l'utilisation de moyennes groupées (moyenne, mode ou médiane par groupe).

Remarque : on réassigne ici la colonne plutôt que d'utiliser `inplace=True`. Avec les versions récentes de Pandas (copy-on-write), `data['Gender'].fillna(..., inplace=True)` ne modifie plus le DataFrame d'origine.

(Pour aller plus loin : [documentation fillna() de Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) & [documentation mode() de Pandas](https://pandas.pydata.org/docs/reference/api/pandas.Series.mode.html))

## # 4 - Comment créer une table pivot avec pivot_table ?

Pandas permet de créer des tableaux croisés dynamiques, comme dans Microsoft Excel. Ici, la colonne clé est `LoanAmount`, qui contient des valeurs manquantes. On peut l'imputer avec le montant moyen de chaque groupe défini par `Gender`, `Married` et `Self_Employed`.

La moyenne de `LoanAmount` pour chaque groupe se calcule ainsi, avec `pivot_table` :

In [ ]:
# Tableau croisé dynamique avec pivot_table
impute_grps = data.pivot_table(values=["LoanAmount"], index=["Gender","Married","Self_Employed"], aggfunc="mean")
print(impute_grps)

(Pour aller plus loin : [documentation pivot_table de Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html))

## # 5 - Qu'est-ce qu'un multi-index et comment l'utiliser ?

Si on regarde le résultat de la technique #4, on remarque une propriété étrange : chaque index est composé d'une combinaison de 3 valeurs. C'est ce qu'on appelle un multi-index. Il permet d'effectuer certaines opérations très rapidement.

En reprenant l'exemple du #4, on a la moyenne de chaque groupe, mais les valeurs manquantes n'ont pas encore été imputées. On peut le faire avec les techniques vues jusqu'ici :

In [ ]:
# Itérer seulement à travers les lignes avec des valeurs manquantes sur la colonne LoanAmount
for i,row in data.loc[data['LoanAmount'].isnull(),:].iterrows():
    ind = tuple([row['Gender'],row['Married'],row['Self_Employed']])
    data.loc[i,'LoanAmount'] = impute_grps.loc[ind].values[0]

# On vérifie les valeurs manquantes:
print(data.apply(num_missing, axis=0))

Remarques :

- Avec un multi-index, `loc` attend un tuple pour désigner un groupe d'index.
- Le suffixe `.values[0]` est nécessaire : sans lui, `loc` renvoie une Series dont l'index ne correspond pas à celui du DataFrame, et l'affectation directe génère une erreur.

## # 6 - Comment croiser deux variables avec crosstab ?

La fonction `crosstab` donne un premier aperçu des données et permet de valider une hypothèse de base.

Ici, on suppose que la colonne `Credit_History` influence fortement le statut du prêt (`Loan_Status`). On peut le tester avec un tableau croisé :

In [ ]:
pd.crosstab(data["Credit_History"],data["Loan_Status"],margins=True)

Ce sont des effectifs absolus, alors que les pourcentages sont plus intuitifs pour une analyse rapide.

On les obtient avec la fonction `apply`, en divisant chaque ligne par son total (la dernière valeur, dans la colonne `All`) :

In [ ]:
def percConvert(ser):
    return ser/float(ser.iloc[-1])

pd.crosstab(data["Credit_History"],data["Loan_Status"],margins=True).apply(percConvert, axis=1)

Il est maintenant évident que les personnes ayant des antécédents de crédit ont beaucoup plus de chances d'obtenir un prêt : 80 % d'entre elles en ont obtenu un, contre seulement 8 % de celles qui n'en ont pas.

Mais ce n'est pas tout : cela raconte aussi une histoire intéressante.

Puisque les antécédents de crédit sont très importants, imaginons que l'on prédise `Y` pour ceux qui en ont, et `N` pour les autres. Étonnamment, cette règle simple est correcte 82 + 378 = 460 fois sur 614, soit 75 % !

Crois-moi, gagner ne serait-ce que quelques dixièmes de point de précision au-delà de ce score est déjà une tâche ardue.

Remarque : ces 75 % sont calculés sur le jeu d'entraînement (`train`). Le jeu de test sera légèrement différent, mais proche.

Cela explique aussi pourquoi un gain de précision infime peut faire gagner des centaines de places au classement d'une compétition Kaggle.

(Pour aller plus loin : [documentation crosstab Pandas](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html))

## # 7 - Comment fusionner des DataFrames avec merge ?

On fusionne deux DataFrames avec la méthode `merge`. La fusion de DataFrames devient essentielle lorsqu'il faut rassembler des informations provenant de différentes sources.

Prenons un cas hypothétique où l'on dispose du prix moyen au mètre carré pour chaque type de zone (`Property_Area`). Définissons un DataFrame :

In [ ]:
prop_rates = pd.DataFrame([1000, 5000, 12000], index=['Rural','Semiurban','Urban'],columns=['rates'])

prop_rates

On peut maintenant fusionner ces informations avec le DataFrame d'origine, avec la méthode `merge` :

In [ ]:
data_merged = data.merge(right=prop_rates, how='inner',left_on='Property_Area',right_index=True, sort=False)

data_merged.pivot_table(values='Credit_History',index=['Property_Area','rates'], aggfunc=len)

Le tableau croisé dynamique confirme que la fusion a réussi. Note que l'argument `values` n'est pas pertinent ici, car on compte simplement les lignes.

(Pour aller plus loin : [documentation merge Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html))

## # 8 - Comment trier un DataFrame selon plusieurs colonnes ?

Pandas permet de trier facilement un DataFrame selon plusieurs colonnes. Voici comment faire :

In [ ]:
data_sorted = data.sort_values(['ApplicantIncome','CoapplicantIncome'], ascending=False)

data_sorted[['ApplicantIncome','CoapplicantIncome']].head(10)

(Pour aller plus loin : [documentation sort_values Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html))

## # 9 - Comment tracer des boîtes à moustaches et des histogrammes avec Pandas ?

Tu ignores peut-être que les boîtes à moustaches et les histogrammes peuvent se tracer directement avec Pandas, sans appeler Matplotlib séparément (il doit simplement être installé). Une seule ligne de code suffit.

Par exemple, pour comparer la distribution de `ApplicantIncome` selon les valeurs de `Loan_Status` :

In [ ]:
data.boxplot(column="ApplicantIncome",by="Loan_Status")

In [ ]:
data.hist(column="ApplicantIncome",by="Loan_Status",bins=30)

Cela montre que le revenu n'est pas un facteur décisif en soi : il n'y a pas de différence appréciable entre les personnes qui ont obtenu le prêt et celles à qui il a été refusé.

(Pour aller plus loin : [documentation hist Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.hist.html) & [documentation boxplot Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.boxplot.html))

## # 10 - Comment regrouper des valeurs numériques avec cut ?

La fonction `pd.cut` regroupe des valeurs numériques en intervalles nommés. Parfois, les valeurs numériques ont plus de sens une fois regroupées.

Par exemple, pour modéliser le trafic (nombre de voitures sur la route) en fonction de l'heure de la journée (en minutes), la minute exacte est peu utile. La période de la journée (« matin », « après-midi », « soirée », « nuit », « nuit tardive ») est bien plus parlante.

Modéliser le trafic de cette manière est plus intuitif et évite le sur-ajustement (*overfitting*).

On définit ici une fonction simple, réutilisable pour regrouper facilement n'importe quelle variable avec `pd.cut` :

In [ ]:
# Regroupement (binning) :
def binning(col, cut_points, labels=None):
    # Définir les valeurs min et max:
    minval = col.min()
    maxval = col.max()

    # Créer une liste en ajoutant min et max à cut_points
    break_points = [minval] + cut_points + [maxval]

    # Si aucun label, on utilise par défaut les labels 0 ... (n-1)
    if not labels:
        labels = range(len(cut_points)+1)

    # Regroupement en utilisant la fonction cut de Pandas
    colBin = pd.cut(col,bins=break_points,labels=labels,include_lowest=True)
    return colBin


cut_points = [90,140,190]
labels = ["low","medium","high","very high"]
data["LoanAmount_Bin"] = binning(data["LoanAmount"], cut_points, labels)
print(data["LoanAmount_Bin"].value_counts(sort=False))

(Pour aller plus loin : [documentation cut Pandas](https://pandas.pydata.org/docs/reference/api/pandas.cut.html))

## # 11 - Comment coder des données nominales avec replace ?

Il arrive souvent qu'il faille modifier les catégories d'une variable nominale, pour plusieurs raisons :

1. Certains algorithmes (comme la régression logistique) exigent des entrées numériques : les variables nominales sont donc généralement codées 0, 1… (n-1).
2. Une même catégorie peut être représentée de deux façons. Par exemple, une température peut être enregistrée comme « High », « Medium », « Low », « H », « low ». Ici, « High » et « H » désignent la même catégorie, et « Low » et « low » ne diffèrent que par la casse, mais Python les lit comme des niveaux différents.
3. Certaines catégories ont des fréquences très basses : il est généralement préférable de les regrouper.

J'ai défini une fonction générique qui prend un dictionnaire en entrée et code les valeurs avec la méthode `replace` de Pandas :

In [ ]:
# Définit une fonction générique en utilisant la fonction replace de Pandas
def coding(col, codeDict):
    colCoded = pd.Series(col, copy=True)
    for key, value in codeDict.items():
        colCoded = colCoded.replace(key, value)
    return colCoded

# Coder LoanStatus avec Y=1 et N=0:
print('Avant codage:')
print(data["Loan_Status"].value_counts())
data["Loan_Status_Coded"] = coding(data["Loan_Status"], {'N':0,'Y':1})
print('\nAprès codage:')
print(data["Loan_Status_Coded"].value_counts())

Les effectifs sont identiques avant et après le codage : notre codage est correct.

(Pour aller plus loin : [documentation replace Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html))

## # 12 - Comment parcourir les lignes d'un DataFrame et corriger les types ?

Ce n'est pas une opération très fréquente, mais mieux vaut savoir la faire pour ne pas rester coincé. Il peut parfois être nécessaire de parcourir toutes les lignes avec une boucle `for`.

Un problème courant est par exemple le mauvais typage des variables en Python. Il se produit généralement lorsque :

1. les variables nominales aux catégories numériques sont traitées comme des nombres ;
2. les variables numériques contenant des caractères dans l'une des lignes (à cause d'une erreur dans les données) sont considérées comme catégorielles.

Il est donc conseillé de définir manuellement les types des colonnes.

Commençons par vérifier les types de toutes les colonnes :

In [ ]:
# Vérifier les types de données:
data.dtypes

On voit ici que `Credit_History` est une variable nominale, mais elle apparaît avec le type `float` (nombre à virgule).

Une bonne solution consiste à créer un fichier CSV listant les noms et les types des colonnes. On peut ensuite écrire une fonction générique qui lit ce fichier et affecte les types. J'ai par exemple créé le fichier `datatypes.csv` (la cellule suivante le crée dans l'environnement) :

In [ ]:
# Création du fichier datatypes.csv : une ligne par colonne, avec son type
types = """feature,type
Loan_ID,ordinal
Gender,categorical
Married,categorical
Dependents,ordinal
Education,categorical
Self_Employed,categorical
ApplicantIncome,continuous
CoapplicantIncome,continuous
LoanAmount,continuous
Loan_Amount_Term,continuous
Credit_History,categorical
Property_Area,categorical
Loan_Status,categorical
"""

with open("datatypes.csv", "w") as f:
    f.write(types)

In [ ]:
# Lecture du fichier:
colTypes = pd.read_csv('datatypes.csv')
print(colTypes)

Une fois le fichier chargé, on parcourt chaque ligne et on attribue le type indiqué dans la colonne `type` à la variable nommée dans la colonne `feature` :

In [ ]:
# Itérer à travers chaque ligne et assigner le type de variable
# Remarque: astype est utilisé pour assigner des types

for i, row in colTypes.iterrows():  # i: dataframe index; row: chaque ligne (row) au format series
    if row['type']=="categorical":
        data[row['feature']]=data[row['feature']].astype(object)
    elif row['type']=="continuous":
        data[row['feature']]=data[row['feature']].astype(float)
print(data.dtypes)

La colonne `Credit_History` est maintenant de type `object`, le type utilisé par Pandas pour représenter les variables nominales.

Remarque : avec Pandas 3, les colonnes de texte sont affichées avec le type `str` et non plus `object`. Les résultats de cet article ont été obtenus avec Pandas 3.0 ; avec une version plus ancienne, tu verras `object` à la place de `str`.

(Pour aller plus loin : [documentation iterrows Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html))